# D166 — Understanding `PERCENT_RANK`

`PERCENT_RANK` shows where a row's rank sits relative to the other rows in its group. MySQL returns a decimal from **0 to 1**. Multiplying by 100 gives a percentage.

```text
PERCENT_RANK = (RANK - 1) / (number of rows - 1)
```

The first-ranked row always receives 0. The last row normally receives 1. Tied values receive the same percent rank because the formula uses `RANK`.

## When is percent rank useful?

Percent rank is useful when the relative position matters more than the original unit. Examples include:

- comparing exam performance across classes of different sizes;
- identifying products or sellers near the top of a distribution;
- comparing salaries across departments;
- placing response times or prices into relative positions; and
- selecting rows above or below a relative cutoff.

It does **not** mean “the percentage of rows below this exact value.” That idea is closer to cumulative distribution, covered in D167.

In [ ]:
import os
import mysql.connector
connection = mysql.connector.connect(
 host=os.environ.get('MYSQL_HOSTNAME','127.0.0.1'),
 port=int(os.environ.get('MYSQL_PORT','3306')),
 user=os.environ.get('MYSQL_USERNAME','root'),
 password=os.environ.get('MYSQL_PASSWORD','root'),
 database=os.environ.get('MYSQL_DATABASE','olist_import_lab'))
print('Connected:', connection.is_connected())

In [ ]:
def execute_sql(sql):
    cursor=connection.cursor(); cursor.execute(sql)
    if not cursor.with_rows:
        connection.commit(); print(f'Affected rows: {cursor.rowcount}'); cursor.close(); return []
    columns=[x[0] for x in cursor.description]; rows=cursor.fetchall(); cursor.close()
    text=[[str(v) for v in row] for row in rows]; widths=[len(c) for c in columns]
    for row in text: widths=[max(w,len(v)) for w,v in zip(widths,row)]
    print(' | '.join(c.ljust(w) for c,w in zip(columns,widths)))
    print('-+-'.join('-'*w for w in widths))
    for row in text: print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))
    return rows

## Example 1 — Exam scores and the formula

The five scores include a tie. Descending order treats the highest score as the first rank. The query displays `RANK`, MySQL's `PERCENT_RANK`, and the same formula calculated manually.

In [ ]:
execute_sql("""
WITH scores AS (
 SELECT 1 student_id, 'Asha' student, 95 score UNION ALL
 SELECT 2, 'Bilal', 90 UNION ALL SELECT 3, 'Chen', 90 UNION ALL
 SELECT 4, 'Divya', 80 UNION ALL SELECT 5, 'Eshan', 60
), ranked AS (
 SELECT *, RANK() OVER(ORDER BY score DESC) rank_value,
        COUNT(*) OVER() row_count, PERCENT_RANK() OVER(ORDER BY score DESC) pr
 FROM scores
)
SELECT student, score, rank_value,
 ROUND(pr*100,1) percent_rank_pct,
 ROUND(100.0*(rank_value-1)/(row_count-1),1) manual_formula_pct
FROM ranked ORDER BY score DESC, student_id
""")

The tied scores of 90 share rank 2 and percent rank 25%. Because `RANK` leaves a gap, the next score receives rank 4 and percent rank 75%.

## Example 2 — Customer response times

For response time, a smaller number is better. Therefore this example orders milliseconds with `ASC`. The fastest response gets 0%, and the slowest gets 100%. Ordering direction changes the business meaning.

In [ ]:
execute_sql("""
WITH responses AS (
 SELECT 'Request A' request_name, 120 milliseconds UNION ALL
 SELECT 'Request B', 180 UNION ALL SELECT 'Request C', 250 UNION ALL
 SELECT 'Request D', 400 UNION ALL SELECT 'Request E', 900 UNION ALL
 SELECT 'Request F', 1500
)
SELECT request_name, milliseconds,
 ROUND(PERCENT_RANK() OVER(ORDER BY milliseconds ASC)*100,1) percent_rank_pct
FROM responses ORDER BY milliseconds
""")

## Example 3 — Compare within each department

`PARTITION BY department` starts a separate calculation for each department. This is useful when groups have different pay scales or different numbers of employees.

In [ ]:
execute_sql("""
WITH salaries AS (
 SELECT 'Data' department,'Asha' employee,90000 salary UNION ALL
 SELECT 'Data','Bilal',70000 UNION ALL SELECT 'Data','Chen',50000 UNION ALL
 SELECT 'Cloud','Divya',120000 UNION ALL SELECT 'Cloud','Eshan',90000 UNION ALL
 SELECT 'Cloud','Fatima',80000 UNION ALL SELECT 'Cloud','Gopal',60000
)
SELECT department, employee, salary,
 ROUND(PERCENT_RANK() OVER(
   PARTITION BY department ORDER BY salary DESC)*100,1) department_percent_rank
FROM salaries ORDER BY department, salary DESC
""")

## Example 4 — Filter a relative range

Window functions are calculated after `WHERE`, so their result cannot normally be filtered in the same query level. A CTE calculates percent rank first. The outer query then keeps the top relative portion.

In [ ]:
execute_sql("""
WITH values_list AS (
 SELECT 10 value_number UNION ALL SELECT 20 UNION ALL SELECT 30 UNION ALL
 SELECT 40 UNION ALL SELECT 50 UNION ALL SELECT 60 UNION ALL SELECT 70 UNION ALL
 SELECT 80 UNION ALL SELECT 90 UNION ALL SELECT 100
), positioned AS (
 SELECT value_number, PERCENT_RANK() OVER(ORDER BY value_number DESC) pr
 FROM values_list
)
SELECT value_number, ROUND(pr*100,1) percent_rank_pct
FROM positioned WHERE pr <= 0.25
ORDER BY value_number DESC
""")

## Olist example — Relative seller position

The CTE first produces one row per seller. `PERCENT_RANK` then compares seller item-value totals. A value near 0 means the seller is near the highest total because the window uses descending order.

In [ ]:
execute_sql("""
WITH seller_totals AS (
 SELECT seller_id, COUNT(*) items_sold, SUM(price) item_value
 FROM olist_order_items GROUP BY seller_id
), positioned AS (
 SELECT *, PERCENT_RANK() OVER(ORDER BY item_value DESC) pr
 FROM seller_totals
)
SELECT seller_id, items_sold, ROUND(item_value,2) item_value,
 ROUND(pr*100,3) percent_rank_pct
FROM positioned
WHERE pr <= 0.01
ORDER BY item_value DESC
""")

## Important points

- The formula uses `RANK`, so ties share a result and create gaps.
- The first row is 0; with more than one row, the last rank is normally 1.
- `ASC` and `DESC` reverse the meaning. State which direction is desirable.
- Use `PARTITION BY` for a separate comparison inside each group.
- Percent rank reports relative ranked position, not cumulative share.
- Calculate it in a CTE before filtering on the result.

In [ ]:
connection.close()
print('MySQL connection closed.')